[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1KpYP47TFSx0SDTaimDfYjXW_7PCBEw63?usp=sharing)

# Interactive Point-Based Image Editing with DragGAN and DragDiffusion

## Introduction
In this seminar, we explore two methods for **interactive point-based image editing**: *DragGAN* and *DragDiffusion*. These techniques allow users to manipulate images by simply clicking and dragging points, offering intuitive control over spatial attributes like pose, shape, and layout.

#### Seminar Plan:

1. **DragGAN** inference: A GAN-based approach for precise image manipulation on the generative image manifold.
2. **DragDiffusion** inference: An extension to diffusion models, enhancing generality and applicability.

---

## Part 1: DragGAN - Interactive Point-Based Manipulation with GANs

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1QS41J34m6TmXj2bWrnkoZJuDZrrRAwIy" width="1000"/>
  <figcaption style="font-style: italic; text-align: center;"></figcaption>
</p>

### 1.1 Why DragGAN?
- **Problem**: Traditional image editing tools (e.g., Photoshop) require manual effort and expertise, while existing generative models (GANs) lack flexible, precise control over spatial attributes.
- **Goal**: Enable users to "drag" any point on a GAN-generated image to a target position, controlling pose, shape, expression, or layout with pixel-level precision.
- **Applications**: Social media edits, movie pre-visualization, car design, etc.
- **Intuition**: Imagine clicking a lion's nose and dragging it to a new spot—DragGAN moves it naturally, changing details (e.g., teeth) as needed, all while staying realistic.

---

### 1.2 How DragGAN Works

The DragGAN method provides an interactive point-based image manipulation technique leveraging a pre-trained StyleGAN2 model.

The overall pipeline consists of two main iterative steps: **motion supervision** and **point tracking**, which are performed repeatedly until the desired image manipulation is achieved.

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1Dbw-m7qtxivFK8gUWv7GmBWCmE6GBYHS" width="1000"/>
  <figcaption style="font-style: italic; text-align: center;">DragGAN Pipeline</figcaption>
</p>

#### Step-by-step Pipeline:

1. **Initialization**:
   - The user generates or selects an initial image from a pre-trained StarGANv2 by providing a latent code $w$.
   - The user then inputs **handle points** (points they want to move) and corresponding **target points** (points where they want the handle points to be placed).
   - Optionally, the user can specify a mask defining which regions of the image should be movable.

2. **Motion Supervision**:
   - This step moves handle points incrementally closer to their target points through latent code optimization.
   - Motion supervision is based on intermediate feature maps (after the 6th block of StyleGAN2).
   - A **shifted feature patch loss** is used, guiding the handle points toward target points incrementally.
   - Formally, the loss is:
   $$
   L = \sum_{i=0}^{n}\sum_{q_i\in \Omega_1(p_i,r_1)}||F(q_i) - F(q_i + d_i)||_1 + \lambda||(F-F_0)\cdot(1-M)||_1
   $$
     where:
     - $F(q)$ are the intermediate GAN features at pixel location $q$.
     - $d_i$ is the direction vector from the handle point to the target point, normalized.
     - $F_0$ are the original features from the initial image.
     - $M$ is the binary mask indicating movable regions.
   - During backpropagation, gradients only flow through the target locations, not the original handle points, ensuring directional motion towards targets.
   - Optimization updates only latent code entries for the first **six layers** of StyleGAN2 to primarily affect spatial attributes, preserving appearance from deeper layers.

3. **Point Tracking**:
   - After each motion supervision optimization, the latent code and resulting image change slightly, necessitating accurate re-localization of handle points.
   - This step uses a nearest neighbor search within GAN feature maps to track the position of each handle point.
   - Formally:
   $$
   p_i := \arg\min_{q_i\in\Omega_2(p_i,r_2)}||F'(q_i)-f_i||_1
   $$
     where:
     - $f_i$ is the original feature vector at the handle point's initial location.
     - $F'$ are the updated feature maps after latent optimization.

4. **Iteration**:
   - Steps 2 and 3 are repeated iteratively (typically 30-200 times) until handle points reach or closely approach their target positions.
   - The process is interactive, allowing users to adjust handle points and targets continuously.

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1_ddE7j3UeuYUmPaTXn8TILDyJ1ZgpMH9" width="800"/>
  <figcaption style="font-style: italic; text-align: center;">Method: Motion Supervision and Point Tracking</figcaption>
</p>

### What Is Trained?

- DragGAN itself does not require explicit training of new networks. Instead, it relies on the pre-trained StyleGAN2, which is already trained on a dataset (FFHQ, LSUN, etc.).
- The latent code $w$ is optimized directly at inference (editing) time—no additional training is performed during the DragGAN manipulation process.

### Training Parameters (Inference-time Optimization):

- **Optimizer**: Adam
- **Learning Rate**: Typically set to $2\times10^{-3}$ or $1\times10^{-3}$, depending on the dataset.
- **Patch radius**:
  - $r_1$ (supervision radius) is typically set around 3 pixels (scaled proportionally with image size).
  - $r_2$ (tracking radius) typically around 12 pixels (scaled proportionally with image size).
- **Regularization** $\lambda$: typically set to 20.
- **Stopping criterion**: The optimization stops when handle points are within a specified pixel distance (typically 1–2 pixels) from target points or reaches a maximum iteration limit (e.g., 300 iterations).

### Inference Procedure:

- At inference time, the method performs interactive edits directly in the latent space.
- User inputs handle points, target points, and optionally masks, and initiates the manipulation.
- Optimization iteratively adjusts latent vectors to move handle points closer to targets while continually tracking handle points through feature-based nearest-neighbor search.
- This iterative inference is efficient (usually taking a few seconds on a GPU like RTX 3090), enabling a real-time interactive editing experience.

### Additional Capabilities:

- The pipeline supports **GAN inversion** for manipulating real images, converting them into latent representations before performing DragGAN manipulations.



### 1.3 Environment Setup


In [ ]:
!git clone https://github.com/Zeqiang-Lai/DragGAN.git

In [ ]:
import sys
sys.path.append(".")
sys.path.append('./DragGAN')

!pip install -q -r DragGAN/requirements.txt

In [ ]:
from gradio_app import main

### 1.4 Run the DragGAN Gradio app

In [ ]:
demo = main()
demo.queue(concurrency_count=1, max_size=20).launch(debug=True)

## Implement from scratch

Restart or delete and reconnect to runtime if needed

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle

from huggingface_hub import hf_hub_download
from transformers import PretrainedConfig, PreTrainedModel, Pipeline

class DragGANConfig(PretrainedConfig):
    model_type = "draggan"

    def __init__(
        self,
        generator_pkl_path=None,
        latent_dim=512,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.generator_pkl_path = generator_pkl_path
        self.latent_dim = latent_dim


In [ ]:
def load_pretrained_stylegan2(generator_pkl_path, device="cpu"):
    """
    Loads a StyleGAN2 (stylegan2-ada-pytorch format) .pkl file.
    """
    import sys
    sys.path.insert(0, '/content/stylegan2-ada-pytorch')  # or local clone path

    with open(generator_pkl_path, 'rb') as f:
        data = pickle.load(f)
    G = data['G_ema'].to(device)
    G.eval()
    return G

In [ ]:
class DragGANModel(PreTrainedModel):
    config_class = DragGANConfig

    def __init__(self, config: DragGANConfig):
        super().__init__(config)

        self.latent_dim = config.latent_dim
        self.device_type = "cuda" if torch.cuda.is_available() else "cpu"

        # Load the pretrained StyleGAN2 generator
        self.stylegan2_generator = load_pretrained_stylegan2(
            config.generator_pkl_path,
            device=self.device_type
        )
        # For demonstration, a trivial external mapping.
        # The real StyleGAN2 does mapping inside G, but let's pretend we do a small transform here.
        self.external_mapping = nn.Sequential(
            nn.Linear(self.latent_dim, self.latent_dim),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, z):
        """
        Forward pass: we take z -> external_mapping -> stylegan2_generator
        """
        w = self.external_mapping(z)  # trivial transform
        # Official stylegan2-ada-pytorch usage: G(z, c, truncation_psi, noise_mode)
        # But if we want to feed w directly to G.synthesis, we need adjustments.
        # We'll do it the simpler way: pass w as "z" to G.
        return self.stylegan2_generator(
            z=w,
            c=None,
            truncation_psi=1.0,
            noise_mode='const'
        )

    @torch.no_grad()
    def generate(self, z):
        return self.forward(z)


In [ ]:
def extract_points_features(image, points):
    B, C, H, W = image.shape
    feats = []
    for (y, x) in points:
        y = max(0, min(H - 1, y))
        x = max(0, min(W - 1, x))
        patch = image[:, :, y:y+1, x:x+1]
        feats.append(patch)
    return torch.cat(feats, dim=2)

class DragGANPipeline(Pipeline):
    """
    Minimal working pipeline that can do:
      1) A normal forward pass (`__call__`).
      2) A separate `drag` method for demonstration.
    """

    def __init__(self, model, device="cpu"):
        # "model" is your DragGANModel or StyleGAN-based model
        super().__init__(model=model)
        self.model = model.to(device)
        self.device = device

    def _sanitize_parameters(self, **pipeline_parameters):
        """
        This method is used to handle & separate pipeline-specific parameters
        from non-pipeline params. Return three dicts:
           (preprocess_params, forward_params, postprocess_params).
        Here we’re ignoring all extra parameters for simplicity.
        """
        return {}, {}, {}

    def preprocess(self, inputs, **preprocess_params):
        """
        Convert raw pipeline input into a model-friendly format.
        The pipeline calls this automatically when you do `pipeline(...)`.
        """
        # Suppose we expect `inputs` to be a latent z
        # We simply return a dict to be passed to `_forward`.
        return {"z": inputs}

    def _forward(self, model_inputs, **forward_params):
        """
        Actually forward through self.model.
        The pipeline calls this after `preprocess`.
        """
        z = model_inputs["z"].to(self.device)
        with torch.no_grad():
            out = self.model.generate(z)
        return out

    def postprocess(self, model_outputs, **postprocess_params):
        """
        Convert the raw model outputs (images) to final pipeline output.
        Here we just return the raw tensor.
        """
        return model_outputs

    def drag(self, z, handle_points, target_points, num_steps=20, lr=0.01):
        z = z.to(self.device)
        z.requires_grad_(True)
        optimizer = torch.optim.Adam([z], lr=lr)

        for step in range(num_steps):
            optimizer.zero_grad()
            out = self.model(z)
            handle_feats = extract_points_features(out, handle_points)
            target_feats = extract_points_features(out, target_points)
            loss = F.l1_loss(handle_feats, target_feats)
            loss.backward()
            optimizer.step()
            if (step+1) % 5 == 0:
                print(f"[drag step {step+1}/{num_steps}] loss={loss.item():.6f}")

        final_img = self.model(z).detach()
        return final_img, z.detach()

    def generate_image(self, z):
        with torch.no_grad():
            z = z.to(self.device)
            return self.model.generate(z)



In [ ]:
%%bash
# 1) Install system dependencies (Ninja, build essentials)
apt-get update -y
apt-get install -y ninja-build build-essential

# 2) Install Python packages
pip install ninja huggingface_hub

# 3) Remove any old stylegan2-ada-pytorch folder, then clone fresh
rm -rf stylegan2-ada-pytorch
git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

In [ ]:
# 1. Download the .pkl file from Hugging Face
#    (Here: "afhqcat.pkl" from "ZeqiangLai/StyleGAN2-pkl" repository)
local_path = hf_hub_download(
    repo_id="ZeqiangLai/StyleGAN2-pkl",
    filename="ada/afhqcat.pkl"
)
print("Downloaded StyleGAN2 model to:", local_path)

# 2. Create a DragGANConfig that points to the local .pkl
config = DragGANConfig(
    generator_pkl_path=local_path,
    latent_dim=512
)

# 3. Build the model and pipeline
model = DragGANModel(config)

pipeline = DragGANPipeline(
    model=model,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Generate an image from random latent
z = torch.randn(1, config.latent_dim)
img_before = pipeline.generate_image(z)
print("Generated image shape (before dragging):", img_before.shape)

# 5. Let's pick some arbitrary handle points and target points.
#    Because afhqcat.pkl is typically 512x512 output, let's pick coords accordingly:
handle_points = [(200, 250), (120, 160)]
target_points = [(400, 450), (320, 410)]

print("Starting the drag operation ...")
img_after, z_after = pipeline.drag(
    z=z,
    handle_points=handle_points,
    target_points=target_points,
    num_steps=20,
    lr=0.01
)
print("Final image shape (after dragging):", img_after.shape)

In [ ]:
import matplotlib.pyplot as plt

img_0to1 = (img_before * 0.5 + 0.5).clamp(0, 1)

img_for_display = img_0to1[0].permute(1, 2, 0).cpu().detach().numpy()

plt.figure(figsize=(6,6))
plt.imshow(img_for_display)
plt.axis("off")
plt.show()

In [ ]:
img_0to1 = (img_after * 0.5 + 0.5).clamp(0, 1)

img_for_display = img_0to1[0].permute(1, 2, 0).cpu().detach().numpy()

plt.figure(figsize=(6,6))
plt.imshow(img_for_display)
plt.axis("off")
plt.show()

___

## Part 2: DragDiffusion - Extending to Diffusion Models

### 2.1 Why DragDiffusion? Overcoming GAN Limitations
- **Problem**: GANs are constrained by their training manifold, limiting edits to synthetic images.
- **Goal**: Extend point-based dragging to real images using diffusion models.
- **Applications**: Photo editing, artistic manipulation.
- **Intuition**: Drag a real dog's ear upward, and the image adapts naturally.

Here's a comprehensive explanation of the **DragDiffusion** model pipeline based on the methodology described:

---

### 2.2 Pipeline Overview:

**DragDiffusion** is an interactive, diffusion-model-based method for point-based image editing, consisting of three main stages:

1. **Identity-preserving Fine-tuning**
2. **Diffusion Latent Optimization**
3. **Inference with Reference-latent-control**

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1d57DtwEDwaMYjUNM5fdic6sy7mYqQ2sL" width="1000"/>
  <figcaption style="font-style: italic; text-align: center;">DragDiffusion Full Pipeline</figcaption>
</p>

---

### 2.3 Identity-preserving Fine-tuning:

**Objective:**  
The model fine-tunes the pretrained Stable Diffusion UNet to better capture and preserve the original image identity.

- **Approach:**
  - Utilizes [**Low-Rank Adaptation (LoRA)**](https://huggingface.co/docs/diffusers/en/training/lora)
  - Injects LoRA layers into the attention modules of UNet.
  - Fine-tuning is performed to minimize noise prediction error for the input image using the loss:
  $$
  L_{ft}(z, \Delta\theta) = E_{\epsilon, t}\left[||\epsilon - \epsilon_{\theta + \Delta\theta}(\alpha_t z + \sigma_t \epsilon)||_2^2\right]
  $$

- **$L_{ft}(z, \Delta\theta)$**:  
  Loss function for fine-tuning UNet with LoRA.

- **$z$**:  
  Input real image latent embedding.

- **$\Delta\theta$**:  
  LoRA parameters (trainable modifications to original UNet weights).

- **$\theta$**:  
  Original UNet parameters (frozen during LoRA fine-tuning).

- **$\epsilon$**:  
  Random noise drawn from a normal distribution $N(0, I)$.

- **$\epsilon_{\theta+\Delta\theta}(\cdot)$**:  
  UNet prediction of noise given noisy latent input (with LoRA weights).

- **$\alpha_t, \sigma_t$**:  
  Parameters controlling the scale of latent and noise at timestep $t$ in the diffusion process.

- **Training Parameters:**
  - Optimizer: **AdamW**
  - Learning rate: **5×10⁻⁴**
  - Batch size: **4**
  - Steps: **80**
  - LoRA rank: **16**

- **Outcome:**
  - Efficiently preserves identity with minimal training time (~25 seconds on an A100 GPU).

---

### 2.4 Diffusion Latent Optimization:

**Objective:**  
Optimizes the diffusion latent based on user-provided points ("handle" points and their "target" locations) to achieve the desired spatial editing.

- **Initial Step:**
  - Applies [**DDIM inversion**](https://huggingface.co/learn/diffusion-course/en/unit4/2) to encode the original image into a diffusion latent at step `t=35`.

- **Iterative Optimization Steps:**
  - Consists of two repeating phases until convergence or iteration limit reached:
    - **Motion Supervision**
    - **Point Tracking**

- **Motion Supervision:**
  - Moves "handle" points toward user-defined "target" points.
  - Loss function includes two terms:
    - Spatial alignment guided by UNet feature maps.
    - Identity preservation by minimizing changes outside masked editable regions.

  The loss is defined as:
  $$
  L_{ms}(\hat{z}_t^k) = \sum_{i=1}^{n}\sum_{q \in \Omega(h_i^k, r_1)}||F_{q+d_i}(\hat{z}_t^k) - sg(F_q(\hat{z}_t^k))||_1 + \lambda ||(\hat{z}_{t-1}^k - sg(z_{t-1}^0))\odot(1-M)||_1
  $$

- **$L_{ms}(\hat{z}_t^k)$**:  
  Motion supervision loss for optimizing the diffusion latent.

- **$\hat{z}_t^k$**:  
  Diffusion latent after $k$-th optimization iteration at timestep $t$.

- **$n$**:  
  Total number of handle-target point pairs provided by the user.

- **$\Omega(h_i^k, r_1)$**:  
  Square patch of radius $r_1$ around the handle point $h_i^k$.

- **$F_{q+d_i}(\hat{z}_t^k)$**:  
  UNet feature at shifted location $q+d_i$, interpolated as needed.

- **$sg(\cdot)$** (stop gradient):  
  Stops gradient computation to prevent updating original features or latents.

- **$F_q(\hat{z}_t^k)$**:  
  UNet feature at position $q$ before shift.

- **$d_i$**:  
  Normalized direction vector from the handle point to its corresponding target point.

- **$\lambda$**:  
  Hyperparameter controlling strength of identity preservation.

- **$\hat{z}_{t-1}^k$**:  
  One-step DDIM denoised latent from optimized latent $\hat{z}_t^k$.

- **$z_{t-1}^0$**:  
  Original latent at timestep $t-1$ (before optimization).

- **Point Tracking:**
  - Updates handle points' positions according to the UNet feature maps after each optimization step.
  - Nearest neighbor search updates the position:
  $$
  h_i^{k+1} = \arg\min_{q\in\Omega(h_i^k,r_2)} ||F_q(\hat{z}_t^{k+1}) - F_{h_i^0}(z_t)||_1
  $$

- **Parameters for optimization:**
  - Optimizer: **Adam**
  - Learning rate: **0.01**
  - Maximum optimization steps: **80**
  - DDIM inversion steps: typically **35**
  - Hyperparameters: `r₁=1`, `r₂=3`, `λ=0.1`

---

### 2.5 Inference with Reference-latent-control:

**Objective:**  
After latent optimization, final denoising is performed using a reference-guided approach to preserve identity and avoid artifacts.

- **Approach:**
  - During inference, [**DDIM denoising**](https://huggingface.co/docs/diffusers/v0.16.0/en/api/schedulers/ddim) is guided by **reference-latent-control**.
  - The self-attention in UNet uses the **key and value** vectors from the original latent (`z_t`) instead of the optimized latent (`\hat{z}_t`), ensuring coherence with the original identity.

- **Result:**
  - Generates high-quality, coherent, edited images preserving original identity and details.

---

### 2.6 Inference Pipeline Summarized:

- Given an input image:
  1. Perform **LoRA fine-tuning** (once per image).
  2. Conduct **DDIM inversion** to obtain latent (`z_t`).
  3. Iteratively optimize latent according to drag instructions.
  4. Perform **reference-latent-control DDIM denoising** to output final edited image.

---

### 2.7 Model Evaluation:

- **Dataset:** [**DRAGBENCH**](https://github.com/Yujun-Shi/DragDiffusion/tree/main/drag_bench_evaluation) (created for evaluating drag-based editing).
- **Metrics:**
  - **Image Fidelity (IF)**: Measures identity preservation (based on LPIPS).
  - **Mean Distance (MD)**: Measures how accurately the dragged points reached their intended targets.

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1MuG5D1mZWqI7kxqrezpLTTIv4vqWezA_" width="1000"/>
  <figcaption style="font-style: italic; text-align: center;">Comparison between DragGAN and DragDiffusion on DRAGBENCH</figcaption>
</p>

---

### 2.8 Computational Efficiency:

- Entire pipeline (fine-tuning → latent optimization → inference) completes quickly:
  - Fine-tuning: ~25 seconds (A100 GPU)
  - Latent optimization: ~10-30 seconds (depending on complexity)
  - Denoising: negligible (1-2 seconds)

---

### 2.9 Limitations and Comparisons:

- Currently, precise tracking of multiple points can occasionally be suboptimal.
- Future improvements aimed at increasing robustness and reliability of multi-point editing.
- Requires fine-tuning, slower than DragGAN.
- **Vs. DragGAN**: Works on real images, more general but less precise on synthetic data.
- **Vs. Traditional Editing**: Automated and generative.

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1kAqpGFSj44ADi9Oa39_tmHeglK25KoWk" width="1000"/>
  <figcaption style="font-style: italic; text-align: center;">Comparison between DragGAN and DragDiffusion</figcaption>
</p>


---

### Environment Setup

In [ ]:
!git clone https://github.com/Yujun-Shi/DragDiffusion.git

In [ ]:
# Restart runtime before running this if needed

# Install specific versions as close to original dragdiff environment as possible
!pip install -q gradio==3.41.1 diffusers==0.24.0 transformers==4.27.0 accelerate==0.17.0

# Install peft version that works with huggingface_hub <=0.20.3
!pip install -q peft==0.4.0 huggingface_hub==0.20.3

# Remaining dependencies
!pip install -q pytorch-lightning==1.5.0

### Run DragDiffusion Gradio App

You may check [GIF](https://github.com/Yujun-Shi/DragDiffusion/blob/main/release-doc/asset/github_video.gif) that demonstrate the usage of UI in a step-by-step manner.

Basically, it consists of the following steps:

### Case 1: Dragging Input Real Images
#### 1) train a LoRA
* Drop our input image into the left-most box.
* Input a prompt describing the image in the "prompt" field
* Click the "Train LoRA" button to train a LoRA given the input image

#### 2) do "drag" editing
* Draw a mask in the left-most box to specify the editable areas.
* Click handle and target points in the middle box. Also, you may reset all points by clicking "Undo point".
* Click the "Run" button to run our algorithm. Edited results will be displayed in the right-most box.

### Case 2: Dragging Diffusion-Generated Images
#### 1) generate an image
* Fill in the generation parameters (e.g., positive/negative prompt, parameters under Generation Config & FreeU Parameters).
* Click "Generate Image".

#### 2) do "drag" on the generated image
* Draw a mask in the left-most box to specify the editable areas
* Click handle points and target points in the middle box.
* Click the "Run" button to run our algorithm. Edited results will be displayed in the right-most box.

In [ ]:
%cd DragDiffusion
!python drag_ui.py

## Implement From Scratch

We will implement DragDiffusion without preliminary LoRA finetuning, you can add it as a homework

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import DiffusionPipeline
from typing import List, Tuple, Optional

class DragDiffusionPipeline(DiffusionPipeline):
    def __init__(
        self,
        vae,
        text_encoder,
        tokenizer,
        unet,
        scheduler
    ):
        super().__init__()
        self.register_modules(
            vae=vae,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            unet=unet,
            scheduler=scheduler
        )

        for p in self.unet.parameters():
            p.requires_grad_(False)
        for p in self.vae.parameters():
            p.requires_grad_(False)
        for p in self.text_encoder.parameters():
            p.requires_grad_(False)

        self.num_inference_steps = 30
        self.guidance_scale = 7.5
        self.drag_steps = 5
        self.drag_alpha = 0.1

    def __call__(
        self,
        prompt: str,
        source_points: List[Tuple[int,int]],
        target_points: List[Tuple[int,int]],
        height: int = 512,
        width: int = 512,
        num_inference_steps: Optional[int] = None,
        guidance_scale: Optional[float] = None,
        enable_drag: bool = False,
        **kwargs
    ):
        device = self._execution_device

        if num_inference_steps is None:
            num_inference_steps = self.num_inference_steps
        if guidance_scale is None:
            guidance_scale = self.guidance_scale

        do_cf_guidance = (guidance_scale > 1.0)

        text_input = self.tokenizer(
            [prompt],
            max_length=77,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)
        text_emb = self.text_encoder(text_input.input_ids)[0]  # [1, 77, hidden_dim]

        if do_cf_guidance:
            uncond_input = self.tokenizer(
                [""],
                max_length=77,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            ).to(device)
            uncond_emb = self.text_encoder(uncond_input.input_ids)[0]  # [1, 77, hidden_dim]
            text_emb = torch.cat([uncond_emb, text_emb], dim=0)        # [2, 77, hidden_dim]

        latents = torch.randn(
            (1, self.unet.config.in_channels, height//8, width//8),
            dtype=torch.float32,
            device=device
        )

        self.scheduler.set_timesteps(num_inference_steps)

        for t in self.scheduler.timesteps:
            if do_cf_guidance:
                # => [2, C, H, W]
                latent_input = torch.cat([latents, latents], dim=0)
            else:
                latent_input = latents

            with torch.no_grad():
                noise_pred = self.unet(latent_input, t, encoder_hidden_states=text_emb).sample

            if do_cf_guidance:
                noise_uncond, noise_text = noise_pred.chunk(2)
                noise_pred = noise_uncond + guidance_scale*(noise_text - noise_uncond)

            latents = self.scheduler.step(noise_pred, t, latents).prev_sample

        #  включили drag
        if enable_drag and len(source_points)==len(target_points)>0:
            latents = self.drag_step(latents, source_points, target_points,
                                     text_emb, do_cf_guidance, guidance_scale)

        # декод
        with torch.no_grad():
            latents = latents / 0.18215
            image = self.vae.decode(latents).sample
            image = (image.clamp(-1,1)+1)/2
        return image

    def drag_step(
        self,
        latents: torch.Tensor,
        source_points: List[Tuple[int,int]],
        target_points: List[Tuple[int,int]],
        text_emb: torch.Tensor,
        do_cf_guidance: bool,
        guidance_scale: float
    ):
        latents = latents.detach().requires_grad_(True)
        device = latents.device

        for _ in range(self.drag_steps):
            t = torch.randint(0, self.scheduler.config.num_train_timesteps, (1,), device=device).long()

            if do_cf_guidance:
                lat_in = torch.cat([latents, latents], dim=0)
            else:
                lat_in = latents

            noise_pred = self.unet(lat_in, t, encoder_hidden_states=text_emb).sample
            if do_cf_guidance:
                noise_u, noise_t = noise_pred.chunk(2)
                noise_pred = noise_u + guidance_scale*(noise_t - noise_u)

            drag_loss = torch.tensor(0., device=device)
            _, _, h, w = latents.shape
            for (sy, sx), (ty, tx) in zip(source_points, target_points):
                sy, sx = min(sy, h-1), min(sx, w-1)
                ty, tx = min(ty, h-1), min(tx, w-1)
                drag_loss += F.mse_loss(latents[0,:,sy,sx], latents[0,:,ty,tx])

            latents.grad = None
            drag_loss.backward()

            with torch.no_grad():
                latents -= self.drag_alpha * latents.grad

        return latents.detach()


In [ ]:
import torch
from diffusers import AutoencoderKL, UNet2DConditionModel, LMSDiscreteScheduler
from transformers import CLIPTokenizer, CLIPTextModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae = AutoencoderKL.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="unet").to(device)
scheduler = LMSDiscreteScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler", prediction_type="epsilon")
text_encoder = CLIPTextModel.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="text_encoder").to(device)
tokenizer = CLIPTokenizer.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="tokenizer")

pipe = DragDiffusionPipeline(
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    unet=unet,
    scheduler=scheduler
).to(device)

### No Drag

In [ ]:
with torch.inference_mode():
    image_no_drag = pipe(
        prompt="A photograph of a cute cat in a hat",
        source_points=[],
        target_points=[],
        height=512,
        width=512,
        num_inference_steps=20,
        guidance_scale=7.5,
        enable_drag=False
    )
print("shape:", image_no_drag.shape)  # [1, 3, H, W]

In [ ]:
from torchvision import transforms
pil_img = transforms.ToPILImage()(image_no_drag.squeeze(0))
pil_img.save("basic_sd_result.png")

### With Drag

In [ ]:
source_pts = [(32, 32)]
target_pts = [(32, 64)]

with torch.enable_grad():
    image_drag = pipe(
        prompt="A cartoon cat on the table",
        source_points=source_pts,
        target_points=target_pts,
        height=256,
        width=256,
        num_inference_steps=20,
        guidance_scale=7.5,
        enable_drag=True
    )


In [ ]:
from torchvision import transforms
pil_img = transforms.ToPILImage()(image_drag.squeeze(0))
pil_img.save("drag_sd_result.png")

## Conclusion
- **DragGAN**: Ideal for synthetic images with high precision.
- **DragDiffusion**: Extends the concept to real images with broader applicability.
- Together, they redefine interactive editing, making it intuitive and powerful.

## Questions (Optional)

### **Conceptual Understanding**
1. **What is the key difference between how DragGAN and DragDiffusion supervise motion from handle points to target points?**
2. **Why is feature space tracking necessary in both DragGAN and DragDiffusion, and how is it implemented differently?**
3. **What are the limitations of using GANs (in DragGAN) that DragDiffusion tries to overcome using diffusion models?**
4. **How does DragDiffusion leverage only one specific diffusion step during latent optimization, and why is this effective?**
5. **Why do both methods separate motion supervision from point tracking in the optimization loop?**

---

### **Technical Details**
6. **In DragDiffusion, what is the purpose of identity-preserving fine-tuning using LoRA, and how does it improve results?**
7. **How does DragGAN determine which layers of StyleGAN2's latent space (W+) to optimize? What does this achieve?**
8. **Why does DragDiffusion rely on UNet features for tracking and supervision instead of traditional optical flow methods?**

---

### **Comparative & Evaluation-Based**
9. **DragGAN uses bilinear interpolation on feature maps for supervision. Why is this design choice important, and how is it mirrored or changed in DragDiffusion?**
10. **How is the latent optimization objective in DragDiffusion structured differently from DragGAN to suit the nature of diffusion models?**
11. **In the ablation study of DragDiffusion, what does the sensitivity to inversion step `t` tell us about the structure of diffusion latent space?**

## References

**Drag Your GAN**: Interactive Point-based Manipulation on the Generative Image Manifold: https://arxiv.org/abs/2305.10973

DragGAN **official** implementation: https://github.com/XingangPan/DragGAN

DragGAN **unofficial** implementation (more intuitive): https://github.com/OpenGVLab/DragGAN

**DragDiffusion**: Harnessing Diffusion Models for Interactive Point-based Image Editing: https://arxiv.org/abs/2306.14435

DragDiffusion official implementation: https://github.com/Yujun-Shi/DragDiffusion

**DDIM Denoising**: https://huggingface.co/docs/diffusers/v0.16.0/en/api/schedulers/ddim

**DDIM Inversion**: https://huggingface.co/learn/diffusion-course/en/unit4/2

**LoRA**: Low-Rank Adaptation of Large Language Models: https://arxiv.org/abs/2106.09685

LoRA finetuning brief explanation with examples from HF: https://huggingface.co/docs/diffusers/en/training/lora

More on GAN inversion: https://github.com/AleKY-G/Awesome-GAN-Inversion

